# 2. The mathematical results (Sections 3 and 4)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Singati2/LOYOLA-PAPER/blob/main/notebooks/02_mathematics.ipynb)

Lemma 3.1, the collision pairs of Theorem 3.5, Example 4.3 and the exhaustive certificates.

Every check prints the value in the paper next to the recomputed value. The last cell fails if any of them differ.

In [1]:
# Setup: runs inside the repository, or on Google Colab (clones the repository first).
import os, sys, subprocess
if os.path.exists("../verify.py"):
    os.chdir("..")
elif not os.path.exists("verify.py"):
    if not os.path.exists("LOYOLA-PAPER"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/Singati2/LOYOLA-PAPER.git"], check=True)
    os.chdir("LOYOLA-PAPER")
if "google.colab" in sys.modules:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt", "pandas", "scikit-learn"], check=True)
ROOT = os.getcwd()
for p in (ROOT, os.path.join(ROOT, "external"), os.path.join(ROOT, "external", "bp"), os.path.join(ROOT, "structural")):
    if p not in sys.path:
        sys.path.insert(0, p)
print("working directory:", ROOT)

working directory: /Users/ganeshshiwakoti/LOYOLA-PAPER


In [2]:
import re, csv, time
import numpy as np, pandas as pd
RESULTS = []

def check(what, paper, got, tol=0.0):
    """Record one comparison between a value printed in the paper and the recomputed value."""
    if isinstance(paper, (int, float)) and not isinstance(paper, bool):
        ok = abs(float(paper) - float(got)) <= tol + 1e-12
    else:
        ok = paper == got
    RESULTS.append({"check": what, "paper": paper, "recomputed": got, "status": "OK" if ok else "MISMATCH"})
    print(f"{'OK      ' if ok else 'MISMATCH'} {what}: paper {paper}, recomputed {got}")
    return ok

def run(cmd, tail=25):
    """Run a repository script, show the end of its output and require exit status 0."""
    t = time.time()
    env = {k: v for k, v in os.environ.items() if k != "MPLBACKEND"}   # Jupyter's inline backend must not leak into scripts
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, env=env)
    out = (r.stdout + r.stderr).strip().splitlines()
    print("\n".join(out[-tail:]))
    print(f"[exit {r.returncode}, {time.time() - t:.0f} s]")
    RESULTS.append({"check": f"script: {cmd}", "paper": "exit 0", "recomputed": f"exit {r.returncode}",
                    "status": "OK" if r.returncode == 0 else "MISMATCH"})
    return r.stdout

### Lemma 3.1: the witness determinant without the pair (1,1)

In [3]:
from mpmath import mp, mpf, log, matrix, det, exp, sqrt
mp.dps = 30
v = lambda i, j: [1, log(i * j), log(i + j), mpf(abs(i - j)) / (i + j)]
d = det(matrix([v(1, 2), v(1, 3), v(2, 2), v(3, 3)]))
closed = -log(mpf(3) / 2) * log(mpf(2) ** 13 / mpf(3) ** 8) / 6
check("det of (1,2),(1,3),(2,2),(3,3)", -0.0150, float(d), 5e-5)
check("closed form -(1/6) ln(3/2) ln(2^13/3^8)", float(d), float(closed), 1e-20)
check("3^8 != 2^13", True, 3 ** 8 != 2 ** 13)

OK       det of (1,2),(1,3),(2,2),(3,3): paper -0.015, recomputed -0.015003225226285728
OK       closed form -(1/6) ln(3/2) ln(2^13/3^8): paper -0.015003225226285728, recomputed -0.015003225226285728
OK       3^8 != 2^13: paper True, recomputed True


True

### Theorem 3.5(iii): the colliding profiles of orders 13 and 16

In [4]:
from fractions import Fraction
from collections import Counter
def qhist(profile):
    h = Counter()
    for (i, j), c in profile.items():
        h[Fraction(abs(i - j), i + j)] += c
    return dict(h)
T1 = {(1,2):1, (1,3):2, (1,6):5, (2,3):2, (2,6):1, (3,3):1}
T2 = {(1,3):3, (1,6):5, (2,2):1, (2,3):2, (3,6):1}
C1 = {(1,2):2, (1,4):8, (2,4):2, (4,4):3}
C2 = {(1,4):8, (2,2):3, (2,4):4}
for name, A, B, n in (("order 13", T1, T2, 13), ("order 16, max degree 4", C1, C2, 16)):
    check(f"{name}: profiles differ", True, A != B)
    check(f"{name}: equal q-histograms", True, qhist(A) == qhist(B))
    check(f"{name}: edges = order - 1", n - 1, sum(A.values()))
print("order-13 histogram:", {str(k): v for k, v in sorted(qhist(T1).items())})

OK       order 13: profiles differ: paper True, recomputed True
OK       order 13: equal q-histograms: paper True, recomputed True
OK       order 13: edges = order - 1: paper 12, recomputed 12
OK       order 16, max degree 4: profiles differ: paper True, recomputed True
OK       order 16, max degree 4: equal q-histograms: paper True, recomputed True
OK       order 16, max degree 4: edges = order - 1: paper 15, recomputed 15
order-13 histogram: {'0': 1, '1/5': 2, '1/3': 1, '1/2': 3, '5/7': 5}


In [5]:
out = run("python structural/minimality_check.py")
m = re.search(r"sharing a q-histogram: (\d+) \(all trees\), (\d+)", out)
check("first collision order, all trees (orders 2-16 enumerated)", 13, int(m.group(1)))
check("first collision order, max degree 4", 16, int(m.group(2)))

all  n= 8 trees=   23 profiles=   21 histograms=   21 colliding classes=0
all  n= 9 trees=   47 profiles=   40 histograms=   40 colliding classes=0
all  n=10 trees=  106 profiles=   79 histograms=   79 colliding classes=0
all  n=11 trees=  235 profiles=  152 histograms=  152 colliding classes=0
all  n=12 trees=  551 profiles=  294 histograms=  294 colliding classes=0
all  n=13 trees= 1301 profiles=  570 histograms=  566 colliding classes=4
all  n=14 trees= 3159 profiles= 1100 histograms= 1090 colliding classes=10
all  n=15 trees= 7741 profiles= 2119 histograms= 2093 colliding classes=26
all  n=16 trees=19320 profiles= 4069 histograms= 3996 colliding classes=69
chem n= 2 trees=    1 profiles=    1 histograms=    1 colliding classes=0
chem n= 3 trees=    1 profiles=    1 histograms=    1 colliding classes=0
chem n= 4 trees=    2 profiles=    2 histograms=    2 colliding classes=0
chem n= 5 trees=    3 profiles=    3 histograms=    3 colliding classes=0
chem n= 6 trees=    5 profiles=    

True

### Example 4.3: the sum-connectivity pair of Rada, Rodriguez and Sigarreta (2022)

In [6]:
D = lambda g: 3 * (exp(g / 2) - 1) - 4 / sqrt(5) * (exp(3 * g / 5) - exp(g / 5)) + 4 / sqrt(6) * (1 - exp(g / 3))
check("D(0) = chi(S) - chi(T)", 0.0, float(D(0)), 1e-25)
check("D(1)", 0.2255, float(D(1)), 5e-5)
check("D(1e-6)", 2.4e-7, float(D(mpf("1e-6"))), 5e-9)
out = run("python structural/sumconn_pair_check.py", tail=6)
check("the order-18 trees themselves give the same D(gamma)", True, "RESULT: Example ex:sumconn confirmed" in out)
check("K4 and K_{1,3} at gamma = ln 4 (Remark 3.9)", 6.0, float(3 * exp(log(4) / 2)), 1e-20)

OK       D(0) = chi(S) - chi(T): paper 0.0, recomputed 0.0
OK       D(1): paper 0.2255, recomputed 0.22553790270840993
OK       D(1e-6): paper 2.4e-07, recomputed 2.401271913096969e-07


chemical trees of order 18: 1725 profiles; pairs with the stated difference: 1
chi(S) - chi(T) = -2.14e-50
gamma = 1.0: D from trees 0.2255379027, closed form 0.2255379027
gamma = 1.0e-6: D from trees 2.401271913e-7, closed form 2.401271913e-7
gamma = -1.0: D from trees -0.2346595942, closed form -0.2346595942
RESULT: Example ex:sumconn confirmed (D(1) = 0.2255, D(1e-6) = 2.4e-7)
[exit 0, 4 s]
OK       the order-18 trees themselves give the same D(gamma): paper True, recomputed True
OK       K4 and K_{1,3} at gamma = ln 4 (Remark 3.9): paper 6.0, recomputed 6.0


True

### Proposition 5.2 and the interval certificates

In [7]:
run("python structural/extremal_trees_check.py", tail=3)
run("python audit/check_mathematics.py", tail=6)

n=12 gamma=+2.0 trees= 551: path unique min/star unique max (gamma>0; exchanged for gamma<0): OK
n=12 gamma=-1.0 trees= 551: path unique min/star unique max (gamma>0; exchanged for gamma<0): OK
RESULT: extremal-tree proposition holds for all orders 4..12
[exit 0, 0 s]


Opposite interval signs: [-11.97, -11.95]
Opposite interval signs: [-2.04, -2.02]
Opposite interval signs: [2.72, 2.74]
Opposite interval signs: [31.12, 31.14]
Five distinct crossings certified, conditional on the stated Laguerre bound.
Mathematics interval checks PASS
[exit 0, 1 s]


'Delta=6: 7525 subsets; exceptions []\nDelta=7: 24129 subsets; exceptions [((1, 7), (2, 6), (3, 5), (4, 4))]\nLemma 3.1 witness without (1,1): determinant -0.0150032252263 != 0 certified\nOpposite interval signs: [-17.18, -17.16]\nOpposite interval signs: [-11.97, -11.95]\nOpposite interval signs: [-2.04, -2.02]\nOpposite interval signs: [2.72, 2.74]\nOpposite interval signs: [31.12, 31.14]\nFive distinct crossings certified, conditional on the stated Laguerre bound.\nMathematics interval checks PASS\n'

## Summary

In [8]:
summary = pd.DataFrame(RESULTS)
display(summary) if "display" in dir() else print(summary.to_string())
n_bad = int((summary.status != "OK").sum())
print(f"{len(summary)} checks, {n_bad} mismatches")
assert n_bad == 0, "some recomputed values differ from the paper"

                                                        check     paper recomputed status
0                              det of (1,2),(1,3),(2,2),(3,3)    -0.015  -0.015003     OK
1                     closed form -(1/6) ln(3/2) ln(2^13/3^8) -0.015003  -0.015003     OK
2                                                 3^8 != 2^13      True       True     OK
3                                   order 13: profiles differ      True       True     OK
4                                order 13: equal q-histograms      True       True     OK
5                                 order 13: edges = order - 1        12         12     OK
6                     order 16, max degree 4: profiles differ      True       True     OK
7                  order 16, max degree 4: equal q-histograms      True       True     OK
8                   order 16, max degree 4: edges = order - 1        15         15     OK
9               script: python structural/minimality_check.py    exit 0     exit 0     OK
10  first 